# Task 3 – Key Management

Assesses the key lifecycle for every cryptographic asset: current key generation, the future PQC key type it should migrate to, a rotation policy driven by data sensitivity, a revocation method, a backup policy driven by where the key is stored, overall PQC readiness, and the TLS upgrade path where relevant.

Reads `reports/risk_report.csv` (produced by `risk_engine.ipynb`) and writes `reports/key_management_report.csv`.

In [ ]:
import pandas as pd
import os

os.makedirs("reports", exist_ok=True)

df = pd.read_csv("reports/risk_report.csv")

print("=" * 60)
print("QuantumSafe Analyzer")
print("Key Lifecycle Management Assessment")
print("=" * 60)

In [ ]:
# -----------------------------
# Current key generation, derived from the actual algorithm + key length
# (not hardcoded, so RSA-2048 vs RSA-4096 assets are labeled correctly)
# -----------------------------
def current_key_generation(algorithm, key_length):
    algorithm = algorithm.upper()
    if algorithm == "RSA":
        return f"RSA-{key_length} Key Pair"
    elif algorithm == "ECC":
        return f"ECC P-{key_length} Key Pair"
    elif algorithm == "AES":
        return f"AES-{key_length} Symmetric Key"
    return "Unknown Key Type"


def future_pqc_key(algorithm):
    algorithm = algorithm.upper()
    if algorithm == "RSA":
        return "ML-KEM Hybrid Key"
    elif algorithm == "ECC":
        return "ML-DSA Hybrid Key"
    elif algorithm == "AES":
        return "AES-256 (No Change)"
    return "Manual Review Required"


def rotation_policy(sensitivity):
    policy = {
        "Critical": "30 Days",
        "High": "90 Days",
        "Medium": "180 Days"
    }
    return policy.get(sensitivity, "365 Days")


def revocation_method(algorithm):
    # Every key type in this inventory is currently certificate/PKI backed,
    # so CRL applies uniformly. Kept as a function (not a constant) so an
    # algorithm-specific method can be added later without touching the caller.
    return "Certificate Revocation List (CRL)"


def backup_policy(key_storage):
    policy = {
        "HSM": "Encrypted HSM Backup",
        "Software Vault": "Encrypted Vault Backup",
        "Cloud KMS": "Cloud Key Backup"
    }
    return policy.get(key_storage, "Manual Backup Review Required")


def pqc_readiness(algorithm):
    algorithm = algorithm.upper()
    if algorithm in ["RSA", "ECC"]:
        return "Migration Required"
    elif algorithm == "AES":
        return "Ready"
    return "Manual Review Required"


def tls_upgrade(tls_version):
    # Only assets that actually negotiate TLS need a hybrid TLS upgrade path.
    if pd.isna(tls_version) or tls_version in ["None", ""]:
        return "N/A"
    return "Hybrid TLS (TLS 1.3 + ML-KEM)"


def infrastructure_impact(algorithm):
    algorithm = algorithm.upper()
    if algorithm in ["RSA", "ECC"]:
        return "Medium"
    return "Low"

In [ ]:
current_key_list = []
future_key_list = []
rotation_list = []
revocation_list = []
backup_list = []
readiness_list = []
tls_list = []
impact_list = []

for _, row in df.iterrows():

    current_key_list.append(current_key_generation(row["Algorithm"], row["KeyLength"]))
    future_key_list.append(future_pqc_key(row["Algorithm"]))
    rotation_list.append(rotation_policy(row["Sensitivity"]))
    revocation_list.append(revocation_method(row["Algorithm"]))
    backup_list.append(backup_policy(row["KeyStorage"]))
    readiness_list.append(pqc_readiness(row["Algorithm"]))
    tls_list.append(tls_upgrade(row["TLSVersion"]))
    impact_list.append(infrastructure_impact(row["Algorithm"]))

    print("\n------------------------------------------")
    print("Asset:", row["Asset"])
    print("Current Key:", current_key_list[-1])
    print("Future PQC Key:", future_key_list[-1])
    print("Rotation Policy:", rotation_list[-1])
    print("PQC Readiness:", readiness_list[-1])

In [ ]:
df["CurrentKeyGeneration"] = current_key_list
df["FuturePQCKey"] = future_key_list
df["RotationPolicy"] = rotation_list
df["RevocationMethod"] = revocation_list
df["BackupPolicy"] = backup_list
df["PQCReadiness"] = readiness_list
df["TLSUpgrade"] = tls_list
df["InfrastructureImpact"] = impact_list

df.to_csv("reports/key_management_report.csv", index=False)

print("\n")
print("=" * 60)
print("Key Management Assessment Completed Successfully")
print("Report Saved:")
print("reports/key_management_report.csv")
print("=" * 60)